<div style="text-align: center;">
  <span style="color: lightskyblue; font-weight: bold; font-size: 50px;">
    Demo File for Pick and Place
  </span>
</div>

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Import Library
</span>

In [1]:
import rby1_sdk as rby
import numpy as np
import rby1_lib_ysw as ysw
import logging
import sys
import time
import pyrealsense2 as rs
import cv2

np.set_printoptions(suppress=True, precision=4, linewidth=300)  

# In .ipynb, show logging 
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    # format="%(asctime)s [%(levelname)s] %(message)s",
    stream=sys.stdout,
    force=True,
)

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Initialization [Connect + PowerON]
</span>

In [2]:
# robotConfig = ysw.setRobotConfig(ysw.MODE_SIM, ysw.MODEL_A)                 # For Simulation
# robotConfig = ysw.setRobotConfig(ysw.MODE_LOCAL_COMPUTER, ysw.MODEL_A)      # For Real Robot + Laptop (RBY1-Hotspot)
robotConfig = ysw.setRobotConfig(ysw.MODE_DIRECT_UPC, ysw.MODEL_A)          # For Real Robot + UPC

robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_AND_POWER)
# robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_ONLY)
if (robot != None):
    model = robot.model()

*************************** [Initialization Starts] ****************************

[INFO] SUCCESS: Connected to the Robot!


[INFO] SUCCESS: Power ON the Robot!
[INFO] SUCCESS: Servo ON the Robot!
[INFO] SUCCESS: Reset the Control Manager!
[INFO] SUCCESS: Enable the Control Manager!
[INFO] SUCCESS: Enable the Grippers!

*************************** [Intialization is Done] ****************************





<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Power OFF
</span>

In [11]:
ysw.powerOFF(robot, robotConfig)

****************************** [powerOFF Starts] *******************************

[INFO] SUCCESS: Disable Grippers!
[INFO] SUCCESS: Disable the Control Manager!

****************************** [PowerOFF is Done] ******************************





True

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Posture Change
</span>

In [3]:
Desired_Pos = ysw.DEMO_POS     # ZERO_POS, READY_POS, PACK_POS, DEMO_POS, DEMO1_POS, DEMO2_POS
ysw.positionCtrl_joint_22dof(robot, Desired_Pos, minimum_time=10.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="color:gray; font-weight:bold; font-size:50px;">
Get Forward Kinematics
</span>

In [17]:
# [Link]:     "link_right_arm_6" 
# -> [Joint]: "tool_right"
# -> [Link]:  "ee_right" 
# -> [Joint]: "gripper_finger_r1"
# -> [Link]:  "ee_finger_r1"    (r1이 ee_right 기준으로 +x 방향, 즉 정면에서 볼때 가까운 방향의 그리퍼임)

# ysw.get_FK(robot, reference_link="base", target_link="link_torso_5")
# ysw.get_FK(robot, reference_link="base", target_link="ee_right")
# ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_right")
# ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_finger_r1")


# End-effector Position Measurement
# fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="link_right_arm_1") 
fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="link_right_arm_5")    # Actual(pitch joint):             0.65 / FK: 0.652    
fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="link_right_arm_6")    # Actual(pitch joint):             0.65 / FK: 0.652 
fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_right")            # Actual(gripper prismatic joint): 0.84 / FK: 0.7781 / Not important
fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_finger_r1")        # Actual(gripper fingertip):       0.9  / FK: 0.8531 /
# "ee_finger_r1" is located at starting point of prsimatic joint of gripper

************* [FK of 'link_right_arm_5' from 'link_head_2' Starts] *************

[Result]

T:
[[ 1.     -0.006   0.0013 -0.0209]
 [ 0.006   1.     -0.0067 -0.2164]
 [-0.0013  0.0067  1.     -0.6522]
 [ 0.      0.      0.      1.    ]]


position [m]: [-0.0209 -0.2164 -0.6522]
rpy [rad]: [0.0067 0.0013 0.006 ]
rpy [deg]: [0.3856 0.0734 0.342 ]

************** [FK of 'link_right_arm_5' from 'link_head_2' Ends] **************



************* [FK of 'link_right_arm_6' from 'link_head_2' Starts] *************

[Result]

T:
[[ 0.9988  0.0489  0.0013 -0.0209]
 [-0.0488  0.9988 -0.0067 -0.2164]
 [-0.0016  0.0067  1.     -0.6522]
 [ 0.      0.      0.      1.    ]]


position [m]: [-0.0209 -0.2164 -0.6522]
rpy [rad]: [ 0.0067  0.0016 -0.0489]
rpy [deg]: [ 0.381   0.0945 -2.8   ]

************** [FK of 'link_right_arm_6' from 'link_head_2' Ends] **************



***************** [FK of 'ee_right' from 'link_head_2' Starts] *****************

[Result]

T:
[[ 0.9988  0.0489  0.0013 -0.0211]
 [

<span style="color:gray; font-weight:bold; font-size:50px;">
[Test] 3D information of colored object
</span>

In [4]:
d435i = ysw.D435i(rgb_on=True, depth_on=True, pixel=(640,480), fps=30)
d435i.initialize()

d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

        detected_objects, color_image, mask = d435i.detectColorObjectContour3D(
            color_image=color_image,
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK,       # Can be changed
            minimum_area=300,                   # Can be changed]
            maximum_depth=1.2                   # ignore distant object larger than x[m] 
        )

        for obj in detected_objects:
            print(
                f"[object {obj['id']}] "
                f"center_3d={np.round(obj['center_3d'], 3)}, "
                f"bbox_3d_size={np.round(obj['bbox_3d_size'], 3)}"
            )

        d435i.imshow("color_image", color_image)
        d435i.imshow("depth_image", depth_colorMap)
        d435i.imshow("mask", mask)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

[INFO] D435i pipeline started
[object 0] center_3d=[-0.006  0.012  0.693], bbox_3d_size=[0.03  0.095 0.077]
[object 1] center_3d=[ 0.482 -0.311  0.948], bbox_3d_size=[0.069 0.213 0.12 ]


QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvid

[object 0] center_3d=[-0.006  0.02   0.698], bbox_3d_size=[0.03  0.093 0.136]
[object 1] center_3d=[ 0.478 -0.31   0.949], bbox_3d_size=[0.069 0.207 0.136]
[object 0] center_3d=[-0.004  0.018  0.71 ], bbox_3d_size=[0.031 0.096 0.118]
[object 1] center_3d=[ 0.48  -0.314  0.948], bbox_3d_size=[0.064 0.207 0.121]
[object 0] center_3d=[-0.006  0.017  0.715], bbox_3d_size=[0.03  0.095 0.083]
[object 1] center_3d=[ 0.478 -0.31   0.945], bbox_3d_size=[0.069 0.214 0.136]
[object 0] center_3d=[-0.007  0.028  0.736], bbox_3d_size=[0.031 0.093 0.098]
[object 1] center_3d=[ 0.48  -0.31   0.943], bbox_3d_size=[0.056 0.211 0.122]
[object 0] center_3d=[-0.007  0.013  0.707], bbox_3d_size=[0.03  0.094 0.146]
[object 1] center_3d=[ 0.475 -0.303  0.942], bbox_3d_size=[0.061 0.208 0.136]
[object 0] center_3d=[-0.007  0.012  0.71 ], bbox_3d_size=[0.03  0.095 0.078]
[object 1] center_3d=[ 0.476 -0.307  0.942], bbox_3d_size=[0.057 0.211 0.139]
[object 0] center_3d=[-0.005  0.013  0.745], bbox_3d_size=[0.031

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Detect the largest object(box) and the second object
</span>

In [5]:
d435i = ysw.D435i(rgb_on=True, depth_on=True, pixel=(640, 480), fps=30)
d435i.initialize()

record_duration = 4.0  # [sec]

largest_samples = []        # Box
second_samples = []         # Object

d435i.startPipeline()

try:
    start_time = time.time()

    while True:
        elapsed_time = time.time() - start_time

        if elapsed_time >= record_duration:
            break

        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)

        detected_objects, color_image, mask = d435i.detectColorObjectContour3D(
            color_image=color_image,
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK,
            minimum_area=300,       
            maximum_depth=1.2,                   # ignore distant object larger than x[m] 
        )

        if len(detected_objects) == 0:
            continue

        detected_objects_sorted = sorted(
            detected_objects,
            key=lambda obj: obj["area_2d"],
            reverse=True
        )

        largest_obj = detected_objects_sorted[0]

        largest_samples.append({
            "area_2d": float(largest_obj["area_2d"]),
            "pixel_center": np.array(largest_obj["pixel_center"], dtype=np.float64),
            "center_3d": np.array(largest_obj["center_3d"], dtype=np.float64),
            "bbox_3d_size": np.array(largest_obj["bbox_3d_size"], dtype=np.float64),
            "bbox_3d_min": np.array(largest_obj["bbox_3d_min"], dtype=np.float64),
            "bbox_3d_max": np.array(largest_obj["bbox_3d_max"], dtype=np.float64),
        })

        if len(detected_objects_sorted) >= 2:
            second_obj = detected_objects_sorted[1]

            second_samples.append({
                "area_2d": float(second_obj["area_2d"]),
                "pixel_center": np.array(second_obj["pixel_center"], dtype=np.float64),
                "center_3d": np.array(second_obj["center_3d"], dtype=np.float64),
                "bbox_3d_size": np.array(second_obj["bbox_3d_size"], dtype=np.float64),
                "bbox_3d_min": np.array(second_obj["bbox_3d_min"], dtype=np.float64),
                "bbox_3d_max": np.array(second_obj["bbox_3d_max"], dtype=np.float64),
            })

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()


largest_avg = ysw.average_detectColorObjectContour3D(largest_samples)
second_avg = ysw.average_detectColorObjectContour3D(second_samples)


print("\n========== Detection Result ==========")

if largest_avg is not None:
    print("\n[1st Largest Object]")
    print(f"num_samples             : {largest_avg['num_samples']}")
    print(f"area_2d_avg             : {largest_avg['area_2d_avg']:.1f}")
    print(f"area_2d_median          : {largest_avg['area_2d_median']:.1f}")
    print(f"pixel_center_avg        : {np.round(largest_avg['pixel_center_avg'], 1)}")
    print(f"pixel_center_median     : {np.round(largest_avg['pixel_center_median'], 1)}")
    print(f"center_3d_avg           : {np.round(largest_avg['center_3d_avg'], 4)} m")
    print(f"center_3d_median        : {np.round(largest_avg['center_3d_median'], 4)} m")
    print(f"bbox_3d_size_avg        : {np.round(largest_avg['bbox_3d_size_avg'], 4)} m")
    print(f"bbox_3d_size_median     : {np.round(largest_avg['bbox_3d_size_median'], 4)} m")
    print(f"bbox_3d_min_median      : {np.round(largest_avg['bbox_3d_min_median'], 4)} m")
    print(f"bbox_3d_max_median      : {np.round(largest_avg['bbox_3d_max_median'], 4)} m")
else:
    print("\n[1st Largest Object]")
    print("No object detected.")

if second_avg is not None:
    print("\n[2nd Largest Object]")
    print(f"num_samples             : {second_avg['num_samples']}")
    print(f"area_2d_avg             : {second_avg['area_2d_avg']:.1f}")
    print(f"area_2d_median          : {second_avg['area_2d_median']:.1f}")
    print(f"pixel_center_avg        : {np.round(second_avg['pixel_center_avg'], 1)}")
    print(f"pixel_center_median     : {np.round(second_avg['pixel_center_median'], 1)}")
    print(f"center_3d_avg           : {np.round(second_avg['center_3d_avg'], 4)} m")
    print(f"center_3d_median        : {np.round(second_avg['center_3d_median'], 4)} m")
    print(f"bbox_3d_size_avg        : {np.round(second_avg['bbox_3d_size_avg'], 4)} m")
    print(f"bbox_3d_size_median     : {np.round(second_avg['bbox_3d_size_median'], 4)} m")
    print(f"bbox_3d_min_median      : {np.round(second_avg['bbox_3d_min_median'], 4)} m")
    print(f"bbox_3d_max_median      : {np.round(second_avg['bbox_3d_max_median'], 4)} m")
else:
    print("\n[2nd Largest Object]")
    print("No second object detected.")

[INFO] D435i pipeline started



========== Detection Result ==========

[1st Largest Object]
num_samples             : 18
area_2d_avg             : 17409.7
area_2d_median          : 17424.5
pixel_center_avg        : [572.1  68.6]
pixel_center_median     : [572.  69.]
center_3d_avg           : [ 0.4093 -0.2748  0.9309] m
center_3d_median        : [ 0.4101 -0.2748  0.9295] m
bbox_3d_size_avg        : [0.2085 0.2322 0.2006] m
bbox_3d_size_median     : [0.1921 0.2296 0.187 ] m
bbox_3d_min_median      : [ 0.3151 -0.3852  0.843 ] m
bbox_3d_max_median      : [ 0.5072 -0.1551  1.0275] m

[2nd Largest Object]
num_samples             : 18
area_2d_avg             : 1887.0
area_2d_median          : 1888.5
pixel_center_avg        : [303.  257.1]
pixel_center_median     : [303. 257.]
center_3d_avg           : [-0.006   0.0121  0.7098] m
center_3d_median        : [-0.006   0.0123  0.704 ] m
bbox_3d_size_avg        : [0.0308 0.0976 0.1159] m
bbox_3d_size_median     : [0.0307 0.0968 0.106 ] m
bbox_3d_min_median      : [-0.0218 -0.03

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Calculation of FK for grasp
</span>

In [6]:
# {A}: vision frame
# {B}: link_head_2 frame
# {M}: Goal frame (object) 
# {M1}: Goal frame (object) before open the gripper 
# {N}: Goal frame (collecting box) 
# {C}: ee_finger_l1/r1 frame
# {D}: final fingertip frame -> same orientation of {C}

box_pos       = largest_avg["center_3d_avg"]
box_3Dsize    = largest_avg["bbox_3d_size_avg"]
object_pos    = second_avg["center_3d_avg"]
object_3Dsize = second_avg["bbox_3d_size_avg"]
object_maxDim = max(object_3Dsize)

desiredPos_linkhead2_ee_before_open_gripper, desiredPos_linkhead2_ee_after_open_gripper = ysw.calculateEEpos_forGrasp(object_pos, object_3Dsize, manual_offset=0.03)

moved_arm = None
if (desiredPos_linkhead2_ee_before_open_gripper[1] <= 0):     moved_arm = "right"
else:     moved_arm = "left"

if (moved_arm == "right"):
    R1 = ysw.rotMat_z(90.0*ysw.DEG2RAD)      # "link_head_2" frame 기준으로 z축 +90도 회전
    R2 = ysw.rotMat_x(-90.0*ysw.DEG2RAD)     # After R1, converted "link_head_2" frame 기준으로 x축 -90도 회전
    R_BC_desired = R1 @ R2                   # Make Grasping orientation same as D435i frame orientation angle (it does not mean that x,y,z axis directions are same)  
elif (moved_arm == "left"):
    R1 = ysw.rotMat_x(90.0*ysw.DEG2RAD)      # "link_head_2" frame 기준으로 x축 +90도 회전
    R2 = ysw.rotMat_y(-90.0*ysw.DEG2RAD)     # After R1, converted "link_head_2" frame 기준으로 y축 -90도 회전
    R_BC_desired = R1 @ R2                   # Make Grasping orientation same as D435i frame orientation angle (it does not mean that x,y,z axis directions are same) 

# [Theoretical] object_pos_from_linkhead2 = ysw.convertPos_vision_to_linkhead2(object_pos) # P_BM
P_BM1 = desiredPos_linkhead2_ee_before_open_gripper
R_BM1 = R_BC_desired
T_BM1 = ysw.make_HT(R_BM1, P_BM1)    

P_BM2 = desiredPos_linkhead2_ee_after_open_gripper
R_BM2 = R_BC_desired
T_BM2 = ysw.make_HT(R_BM2, P_BM2)    

# "ee_finger_l1/r1" to final end-effector -> should be physically measured
pos_from_eeFinger_to_fingertip = np.array([0, 0, ysw.GRIPPER_Z_OFFSET_FROM_LINK_EE_FINGER]) # P_CD
P_CD = pos_from_eeFinger_to_fingertip
T_CD = ysw.make_HT(np.eye(3), P_CD)    

T_BC_target_before_open_gripper = T_BM1 @ np.linalg.inv(T_CD)
T_BC_target_after_open_gripper  = T_BM2 @ np.linalg.inv(T_CD)
print(f"Before open gripper: \n{T_BC_target_before_open_gripper}\n")
print(f"After open gripper: \n{T_BC_target_after_open_gripper}\n")
print(f"Moving Arm: {moved_arm}")


Before open gripper: 
[[ 0.      0.     -1.      0.5613]
 [-1.      0.     -0.      0.0285]
 [ 0.      1.      0.      0.0579]
 [ 0.      0.      0.      1.    ]]

After open gripper: 
[[ 0.      0.     -1.      0.6793]
 [-1.      0.     -0.      0.0285]
 [ 0.      1.      0.      0.0579]
 [ 0.      0.      0.      1.    ]]

Moving Arm: left


<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Move to the position (until before grasp position)
</span>

In [7]:
### Move to Pre-posture (Joint Position Control)
pre_posture = ysw.DEMO_POS     # ZERO_POS, READY_POS, PACK_POS, DEMO_POS
ysw.positionCtrl_joint_22dof(robot, pre_posture, minimum_time=5.0, control_hold_time=0.5)


pos_before_open_gripper = T_BC_target_before_open_gripper[:3,3]
ori_before_open_gripper = ysw.rotMat_to_rpy(T_BC_target_before_open_gripper[:3,:3])

if (moved_arm == "right"):      # Object is in right side of robot
    ### Cartesian Position Control example (Whole-Body version)
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_head_2",
                                                "target_link": "ee_finger_r1",
                                                "pos": pos_before_open_gripper,                  
                                                "rpy": ori_before_open_gripper,                    
                                            },                 
                                        ]
elif (moved_arm == "left"):       # Object is in left side of robot
    ### Cartesian Position Control example (Whole-Body version)
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_head_2",
                                                "target_link": "ee_finger_l1",
                                                "pos": pos_before_open_gripper,                  
                                                "rpy": ori_before_open_gripper,                    
                                            },                 
                                        ]

# [rv = "function"] form is Necessary when "get_return=False"
rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    # joint_targets=joint_targets_sample, 
    joint_targets=None,
    # control_hold_time=0.5,
    get_return=True
)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************



*************** [Cartesian Position Control(Whole-Body) Starts] ****************

[INFO] Add (Whole-Body) Cartesian target 0: 'link_head_2'->'ee_finger_l1', pos=[0.5613 0.0285 0.0579], rpy=[ 1.5708 -0.     -1.5708]

**************** [Cartesian Position Control(Whole-Body) Ends] *****************





<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Open gripper before grasp
</span>

In [8]:
object_meanDim = np.mean(second_avg["bbox_3d_size_avg"])
desiredGripperPos = object_meanDim 

gripper = ysw.Gripper()
gripper.connect(device_name=ysw.GRIPPER_DEVICE_NAME, protocol_version=ysw.DXL_PROTOCOL_VERSION, baudrate=ysw.DXL_BAUDRATE)
gripper.test_ping()

gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_R, operation_mode=ysw.DXL_CURRENT_BASED_POSITION_CONTROL_MODE)
gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_L, operation_mode=ysw.DXL_CURRENT_BASED_POSITION_CONTROL_MODE)

# desiredGripperPos_mm = desiredGripperPos * 1000.0 + ysw.GRIPPER_SAFETY_OPEN_OFFSET
desiredGripperPos_mm = desiredGripperPos * 1000.0
if desiredGripperPos_mm >= ysw.GRIPPER_POS_RANGE_LINEAR:
    desiredGripperPos_mm = ysw.GRIPPER_POS_RANGE_LINEAR

if (moved_arm == "right"):      # Object is in right side of robot
    gripper.current_based_position_control_linear(ysw.GRIPPER_ID_R, current_ampere_mag=0.02, linear_pos=desiredGripperPos_mm)
    time.sleep(1)
    gripper.disable(ysw.GRIPPER_ID_R)
elif (moved_arm == "left"):       # Object is in left side of robot
    gripper.current_based_position_control_linear(ysw.GRIPPER_ID_L, current_ampere_mag=0.02, linear_pos=desiredGripperPos_mm)
    time.sleep(1)
    gripper.disable(ysw.GRIPPER_ID_L)

************************* [Gripper Connection Starts] **************************

[INFO] SUCCESS: open the port: /dev/rby1_gripper
[INFO] SUCCESS: change the baudrate 2000000

************************* [Gripper Connection is Done] *************************



************************** [Gripper Ping Test Starts] **************************

[INFO] [DXL ID:0] SUCCESS: Dynamixel model number is 1030
[INFO] [DXL ID:1] SUCCESS: Dynamixel model number is 1030

************************* [Gripper Ping Test is Done] **************************



********************* [Gripper [DXL ID:0] setting Starts] **********************

[INFO] SUCCESS: [DXL ID:0] Torque is Disabled
[INFO] SUCCESS: [DXL ID:0] Set Current-based Position Control Mode
[INFO] SUCCESS: [DXL ID:0] Torque is Enabled

********************* [Gripper [DXL ID:0] setting is Done] *********************



********************* [Gripper [DXL ID:1] setting Starts] **********************

[INFO] SUCCESS: [DXL ID:1] Torque is Disabled
[INF

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Move EE to remained distance after gripper open
</span>

In [9]:
pos_after_open_gripper = T_BC_target_after_open_gripper[:3,3]
ori_after_open_gripper = ysw.rotMat_to_rpy(T_BC_target_after_open_gripper[:3,:3])

if (moved_arm == "right"):      # Object is in right side of robot
    ### Cartesian Position Control example (Whole-Body version)
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_head_2",
                                                "target_link": "ee_finger_r1",
                                                "pos": pos_after_open_gripper,                  
                                                "rpy": ori_after_open_gripper,                    
                                            },                 
                                        ]
    
elif (moved_arm == "left"):       # Object is in left side of robot
    ### Cartesian Position Control example (Whole-Body version)
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_head_2",
                                                "target_link": "ee_finger_l1",
                                                "pos": pos_after_open_gripper,                  
                                                "rpy": ori_after_open_gripper,                    
                                            },                 
                                        ]


# [rv = "function"] form is Necessary when "get_return=False"
rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    # joint_targets=joint_targets_sample, 
    joint_targets=None,
    # control_hold_time=0.5,
    get_return=True
)

*************** [Cartesian Position Control(Whole-Body) Starts] ****************

[INFO] Add (Whole-Body) Cartesian target 0: 'link_head_2'->'ee_finger_l1', pos=[0.6793 0.0285 0.0579], rpy=[ 1.5708 -0.     -1.5708]

**************** [Cartesian Position Control(Whole-Body) Ends] *****************





<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Close gripper to grasp
</span>

In [10]:
object_minDim = min(second_avg["bbox_3d_size_avg"])
desiredGripperPos = object_minDim 

desiredGripperPos_mm = desiredGripperPos * 1000.0 + ysw.GRIPPER_SAFETY_CLOSE_OFFSET
if desiredGripperPos_mm >= ysw.GRIPPER_POS_RANGE_LINEAR:
    desiredGripperPos_mm = ysw.GRIPPER_POS_RANGE_LINEAR

if (moved_arm=="right"):      # Object is in right side of robot
    gripper.enable(ysw.GRIPPER_ID_R)
    gripper.current_based_position_control_linear(ysw.GRIPPER_ID_R, current_ampere_mag=0.2, linear_pos=desiredGripperPos_mm)
elif (moved_arm=="left"):       # Object is in left side of robot
    gripper.enable(ysw.GRIPPER_ID_L)
    gripper.current_based_position_control_linear(ysw.GRIPPER_ID_L, current_ampere_mag=0.2, linear_pos=desiredGripperPos_mm)

********************** [Gripper [DXL ID:1] Enable Starts] **********************

[INFO] SUCCESS: [DXL ID:1] Torque is Enabled

********************* [Gripper [DXL ID:1] Enable is Done] **********************



********** [Gripper [DXL ID:1] Current-based Position Control Starts] **********

Desired Linear Position = 18.783138289633726mm
[INFO] [DXL ID:1] SUCCESS: operation mode = 5


[INFO] SUCCESS: [DXL ID:1] Set Current [0.1345 A]
[INFO] SUCCESS: [DXL ID:1] Set Goal Position [Tick]: signed=1413, uint32=1413

*********** [Gripper [DXL ID:1] Current-based Position Control Ends] ***********





In [11]:
### Move to Pre-posture (Joint Position Control)
pre_posture = ysw.DEMO_POS     # ZERO_POS, READY_POS, PACK_POS, DEMO_POS
ysw.positionCtrl_joint_22dof(robot, pre_posture, minimum_time=5.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Move to Large Box
</span>

In [ ]:
box_pos = largest_avg["center_3d_avg"]
box_3Dsize = largest_avg["bbox_3d_size_avg"]
box_maxDepth = max(object_3Dsize)




            
desiredPos_linkhead2_eefinger_before_open_gripper, desiredPos_linkhead2_eefinger_after_open_gripper = ysw.calculateEEpos_forGrasp(object_pos, object_3Dsize)

if (desiredPos_linkhead2_eefinger_before_open_gripper[1] <= 0):      # Object is in right side of robot
    fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_finger_r1")
    pos_original = fk_result["position"]
    rpy_original = fk_result["rpy"]
     
    R1 = ysw.rotMat_z(90.0*ysw.DEG2RAD)      # "link_head_2" frame 기준으로 z축 +90도 회전
    R2 = ysw.rotMat_x(-90.0*ysw.DEG2RAD)     # After R1, converted "link_head_2" frame 기준으로 x축 -90도 회전
    R = R1 @ R2
    rpy_changed = ysw.rotMat_to_rpy(R)
    
    ### Cartesian Position Control example (Whole-Body version)
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_head_2",
                                                "target_link": "ee_finger_r1",
                                                "pos": desiredPos_linkhead2_eefinger_before_open_gripper,                  
                                                "rpy": rpy_changed,                    
                                            },                 
                                        ]
    
else:       # Object is in left side of robot
    fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_finger_l1")    
    pos_original = fk_result["position"]
    rpy_original = fk_result["rpy"]
    
    R1 = ysw.rotMat_x(90.0*ysw.DEG2RAD)    # "link_head_2" frame 기준으로 x축 +90도 회전
    R2 = ysw.rotMat_y(-90.0*ysw.DEG2RAD)    # "link_head_2" frame 기준으로 y축 -90도 회전
    R = R1 @ R2
    rpy_changed = ysw.rotMat_to_rpy(R)

    ### Cartesian Position Control example (Whole-Body version)
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_head_2",
                                                "target_link": "ee_finger_l1",
                                                "pos": desiredPos_linkhead2_eefinger_before_open_gripper,                  
                                                "rpy": rpy_changed,                    
                                            },                 
                                        ]


# [rv = "function"] form is Necessary when "get_return=False"
rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    # joint_targets=joint_targets_sample, 
    joint_targets=None,
    # control_hold_time=0.5,
    get_return=True
)

In [12]:
gripper.enable(ysw.GRIPPER_ID_R)
gripper.current_based_position_control_linear(ysw.GRIPPER_ID_R, current_ampere_mag=0.2, linear_pos=80)

********************** [Gripper [DXL ID:0] Enable Starts] **********************

[INFO] SUCCESS: [DXL ID:0] Torque is Enabled

********************* [Gripper [DXL ID:0] Enable is Done] **********************



********** [Gripper [DXL ID:0] Current-based Position Control Starts] **********

Desired Linear Position = 80mm
[INFO] [DXL ID:0] SUCCESS: operation mode = 5
[INFO] SUCCESS: [DXL ID:0] Set Current [0.1345 A]
[INFO] SUCCESS: [DXL ID:0] Set Goal Position [Tick]: signed=-1581, uint32=4294965715

*********** [Gripper [DXL ID:0] Current-based Position Control Ends] ***********





True

In [13]:
fk_result = ysw.getState(robot)
fk_result

***************************** [Robot State Starts] *****************************

RobotState_A(
  timestamp=datetime.datetime(1970, 1, 1, 6, 37, 34, 608040),
  system_stat=SystemStat(
    cpu_usage=0.0422,
    memory_usage=0.0665,
    uptime=2.39e+04,
    program_uptime=2.38e+04
  ),
  battery_state=BatteryState(
    voltage=56,
    current=3.5,
    level_percent=72
  ),
  power_states=[PowerState(
    state=PowerState.State.PowerOn,
    voltage=0
  ), PowerState(
    state=PowerState.State.PowerOn,
    voltage=0
  ), PowerState(
    state=PowerState.State.PowerOn,
    voltage=0
  ), PowerState(
    state=PowerState.State.PowerOn,
    voltage=56
  )],
  emo_states=[EMOState(
    state=EMOState.State.Released
  )],
  joint_states=[JointState(
    time_since_last_update=datetime.timedelta(microseconds=2),
    is_ready=True,
    fet_state=JointState.FETState.On,
    run_state=JointState.RunState.ControlOn,
    init_state=JointState.InitializationState.Initialized,
    motor_type=1,
    mo

RobotState_A(
  timestamp=datetime.datetime(1970, 1, 1, 6, 37, 34, 608040),
  system_stat=SystemStat(
    cpu_usage=0.0422,
    memory_usage=0.0665,
    uptime=2.39e+04,
    program_uptime=2.38e+04
  ),
  battery_state=BatteryState(
    voltage=56,
    current=3.5,
    level_percent=72
  ),
  power_states=[PowerState(
    state=PowerState.State.PowerOn,
    voltage=0
  ), PowerState(
    state=PowerState.State.PowerOn,
    voltage=0
  ), PowerState(
    state=PowerState.State.PowerOn,
    voltage=0
  ), PowerState(
    state=PowerState.State.PowerOn,
    voltage=56
  )],
  emo_states=[EMOState(
    state=EMOState.State.Released
  )],
  joint_states=[JointState(
    time_since_last_update=datetime.timedelta(microseconds=2),
    is_ready=True,
    fet_state=JointState.FETState.On,
    run_state=JointState.RunState.ControlOn,
    init_state=JointState.InitializationState.Initialized,
    motor_type=1,
    motor_state=7,
    power_on=True,
    position=0.364,
    velocity=0,
    current=0.